In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import sqlite3
import random
import holidays

fake = Faker('pt_BR')
conn = sqlite3.connect('projetoecommerce.db')
cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE IF NOT EXISTS dim_canais_venda(
    id_canal int primary key,
    nome_canal varchar (25) not null,
    tipo_canal varchar(20) not null,
    percentual_comissao decimal(5,2) default 0,
    constraint chk_nome_canal check (nome_canal in ('Site', 'App', 'Whatsapp', 'Marketplace')),
    constraint chk_tipo_canal check (tipo_canal in ('Direto', 'Indireto'))
);

CREATE TABLE IF NOT EXISTS dim_status_pedido(
    id_status int primary key,
    status varchar (25) not null,
    pedido_concluido int not null,
    constraint chk_status check (status in ('Aguardando Pagamento', 'Pagamento realizado', 'Em Processamento', 'Enviado', 'Entregue', 'Cancelado', 'Estornado')),
    constraint chk_pedido_concluido check (pedido_concluido in (0, 1))
);

CREATE TABLE IF NOT EXISTS dim_produtos(
    id_produto int primary key,
    nome_produto varchar (100) not null,
    categoria varchar (25) not null,
    subcategoria varchar(100) not null,
    marca varchar (100) not null,
    custo_unitario int not null
);

CREATE TABLE IF NOT EXISTS dim_clientes(
    id_cliente int primary key,
    nome_completo varchar (100) not null,
    genero varchar (25) not null,
    data_nascimento date not null,
    email varchar (255) unique not null,
    cidade varchar (100) not null, 
    estado char (2) not null
);

CREATE TABLE IF NOT EXISTS dim_calendario(
    id_data date primary key,
    ano int not null,
    mes char(2) not null,
    trimestre int not null,
    dia_semana varchar(20) not null,
    flag_feriado int not null
);

CREATE TABLE IF NOT EXISTS fato_vendas(
    id_venda int primary key,
    id_cliente int not null references dim_clientes(id_cliente),
    id_produto int not null references dim_produtos(id_produto),
    id_data date not null references dim_calendario(id_data),
    id_canal int not null references dim_canais_venda(id_canal), 
    id_status int not null references dim_status_pedido(id_status),
    quantidade int not null,
    valor_unitario decimal (10,2) not null,
    valor_total decimal (10,2) not null,
    valor_desconto decimal (10,2) not null
);
""")

NUM_CLIENTES = 3000
NUM_PRODUTOS = 150
NUM_VENDAS = 50000

canais_data = [
    (1, 'Site', 'Direto', 0.0),
    (2, 'App', 'Direto', 0.0),
    (3, 'Whatsapp', 'Indireto', 5.0),
    (4, 'Marketplace', 'Indireto', 15.0)
]
cursor.executemany("INSERT OR IGNORE INTO dim_canais_venda VALUES (?,?,?,?)", canais_data)

status_data = [
    (1, 'Aguardando Pagamento', 0), (2, 'Pagamento realizado', 1),
    (3, 'Em Processamento', 1), (4, 'Enviado', 1),
    (5, 'Entregue', 1), (6, 'Cancelado', 0), (7, 'Estornado', 0)
]
cursor.executemany("INSERT OR IGNORE INTO dim_status_pedido VALUES (?,?,?)", status_data)

clientes = []
estados = ['AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO']
for i in range(1, NUM_CLIENTES + 1):
    clientes.append([
        i, fake.name(), random.choice(['Masculino', 'Feminino', 'Outro', 'Prefiro não informar']),
        fake.date_of_birth(minimum_age=18, maximum_age=80).strftime('%Y-%m-%d'),
        fake.unique.email(), fake.city(), random.choice(estados)
    ])
df_clientes = pd.DataFrame(clientes, columns=['id_cliente', 'nome_completo', 'genero', 'data_nascimento', 'email', 'cidade', 'estado'])

produtos = []
marcas = ['Sony', 'Samsung', 'Dell', 'Bosch', 'Mattel', 'Pirelli', 'Livraria ABC', 'Móveis Design']
for i in range(1, NUM_PRODUTOS + 1):
    cat = random.choice(['Eletrônicos', 'Móveis', 'Livros', 'Ferramentas', 'Brinquedos', 'Automotivos'])
    custo = random.randint(50, 2000)
    produtos.append([
        i, f"Produto {fake.word().upper()} {i}", cat, f"Subcat {cat}", random.choice(marcas), custo
    ])
df_produtos = pd.DataFrame(produtos, columns=['id_produto', 'nome_produto', 'categoria', 'subcategoria', 'marca', 'custo_unitario'])

datas = pd.date_range(start='2024-01-01', end='2025-12-31')
br_holidays = holidays.Brazil() 

mapa_dias = {
    'Monday': 'Segunda-feira', 'Tuesday': 'Terça-feira', 
    'Wednesday': 'Quarta-feira', 'Thursday': 'Quinta-feira', 
    'Friday': 'Sexta-feira', 'Saturday': 'Sábado', 'Sunday': 'Domingo'
}

flags = [1 if d in br_holidays else 0 for d in datas]

df_calendario = pd.DataFrame({
    'id_data': datas.strftime('%Y-%m-%d'),
    'ano': datas.year,
    'mes': datas.strftime('%m'),
    'trimestre': datas.quarter,
    'dia_semana': datas.day_name().map(mapa_dias),
    'flag_feriado': flags 
})

vendas_ids_p = np.random.randint(1, NUM_PRODUTOS + 1, size=NUM_VENDAS)
df_vendas = pd.DataFrame({
    'id_venda': range(1, NUM_VENDAS + 1),
    'id_cliente': np.random.randint(1, NUM_CLIENTES + 1, size=NUM_VENDAS),
    'id_produto': vendas_ids_p,
    'id_data': np.random.choice(df_calendario['id_data'], size=NUM_VENDAS),
    'id_canal': np.random.randint(1, 5, size=NUM_VENDAS),
    'id_status': np.random.choice([1, 2, 3, 4, 5, 6, 7], size=NUM_VENDAS, p=[0.05, 0.1, 0.1, 0.1, 0.55, 0.05, 0.05]),
    'quantidade': np.random.randint(1, 6, size=NUM_VENDAS)
})

df_vendas = df_vendas.merge(df_produtos[['id_produto', 'custo_unitario']], on='id_produto')
df_vendas['valor_unitario'] = (df_vendas['custo_unitario'] * 1.5).round(2) 
df_vendas['valor_total'] = (df_vendas['valor_unitario'] * df_vendas['quantidade']).round(2)
df_vendas['valor_desconto'] = (df_vendas['valor_total'] * np.random.choice([0, 0.05, 0.1], size=NUM_VENDAS, p=[0.7, 0.2, 0.1])).round(2)

df_clientes.to_sql('dim_clientes', conn, if_exists='append', index=False)
df_produtos.to_sql('dim_produtos', conn, if_exists='append', index=False)
df_calendario.to_sql('dim_calendario', conn, if_exists='append', index=False)
df_vendas.drop(columns=['custo_unitario']).to_sql('fato_vendas', conn, if_exists='append', index=False)

conn.commit()
conn.close()
print("Banco de dados 'projetoecommerce.db' atualizado com sucesso!")

Banco de dados 'projetoecommerce.db' atualizado com sucesso!
